# Shell-Domain Scaling Framework — Supplemental Notebook

**Paper:** Shell-Domain Scaling: An Integer-Only Framework for Eliminating Floating-Point Failure Modes

**Author:** Kenneth John Rumments

**ORCID:** 0009-0001-0924-1850

**Zenodo DOI:** 10.5281/zenodo.20289163

**Companion paper:** Root Shell Mathematics: An Integer-Only Method for Integer k-th Root Bracketing and Refinement

**Companion DOI:** 10.5281/zenodo.20291013

## Purpose

This notebook provides executable supplemental demonstrations for the Shell-Domain Scaling paper. The tests compare IEEE 754 floating-point behavior with integer-scaled SDS representations across common numerical failure modes.

The notebook includes demonstrations of rounding drift, decimal equality failure, catastrophic cancellation, reversible arithmetic behavior, subnormal collapse, logarithmic boundary error, fractional exponent behavior, and representation-dependent trajectory drift near a sensitive Mandelbrot boundary.

## Scope

SDS does not claim to represent arbitrary real numbers exactly with finite integers. Exactness applies to values representable inside the chosen integer-scaled domain.

Addition and subtraction are exact under compatible scaling and sufficient integer capacity. Multiplication may require denominator growth, exact rescaling, or domain-specific handling. Reversibility applies to operations that are invertible within the chosen integer representation.

The Mandelbrot boundary test demonstrates representation-dependent trajectory divergence near a sensitive boundary. It does not claim that SDS prevents mathematical escape for points outside the set.

## Runtime

This notebook is designed for standard Python 3 and Google Colab. No GPU is required.

## License

Notebook/code: Apache License 2.0
Paper text: Creative Commons Attribution 4.0 International


In [1]:
# Shell-Domain Scaling Framework — Supplemental Notebook
# Kenneth John Rumments
# DOI: 10.5281/zenodo.20289163
# Companion DOI: 10.5281/zenodo.20291013
#
# License for notebook/code: Apache License 2.0
# Python: 3.x
#
# Standard-library imports only

from decimal import Decimal, getcontext
from fractions import Fraction
import math
import time

def rumments_root(x: int, k: int) -> int:
    """
    Computes the integer r such that r^k ≤ x < (r+1)^k,
    using fast bit-length estimation for bracketing.

    Works for arbitrarily large integers. No floats used.
    """
    if x < 0 or k < 1:
        raise ValueError("x must be non-negative and k ≥ 1")
    if x == 0:
        return 0
    if x == 1:
        return 1

    # Estimate r ≈ floor(x^(1/k)) using bit-length trick
    est_bitlen = (x.bit_length() - 1) // k
    r = 1 << est_bitlen  # r = 2^floor(bitlen(x)/k)

    # Set tight binary search bounds around r
    low = max(0, r >> 1)
    high = (r << 1) + 4  # Slight overestimate for safety

    # Standard binary search
    while low < high:
        mid = (low + high) // 2
        p = pow(mid, k)
        if p <= x:
            low = mid + 1
        else:
            high = mid

    return low - 1


def rumments_fstroot(x: int, k: int, w: int = 32) -> int:
    """
    Exact floor k-th root using radix-2^w chunked bit assembly.
    - Integer-only, no Newton, no floats.
    - ~ ceil( (bitlen(x)/k) / w ) capped-power checks.
    """
    if x < 0 or k < 1:
        raise ValueError("x must be non-negative and k ≥ 1")
    if x in (0, 1) or k == 1:
        return x

    # Initial MSB position for the root
    a = (x.bit_length() - 1) // k
    if a <= 0:
        # Tiny roots: fall back safely
        r = 0
        s = 0
        while True:
            t = r + (1 << s)
            if _pow_leq(t, k, x):
                r = t
            if s == 0:
                break
            s -= 1
        return r

    # Start at the highest chunk boundary
    s = (a // w) * w
    r = 0

    while True:
        t = r + (1 << s)
        if _pow_leq(t, k, x):
            r = t
        if s == 0:
            break
        s -= w
        if s < 0:
            s = 0  # last partial chunk

    # Final single tighten
    if _pow_leq(r + 1, k, x):
        r += 1
    return r


def rumments_integer_log_base(x: int, k: int) -> int:
    """Returns floor(log_k x) using integer-only root shell logic."""
    if x < 1 or k < 2:
        raise ValueError("x must be ≥ 1 and k must be ≥ 2")
    # Bracket log using exponential shell
    low = 0
    high = 1
    while pow(k, high) <= x:
        high *= 2  # fast skip ahead
    # Binary search for floor(log_k x)
    while low < high:
        mid = (low + high) // 2
        p = pow(k, mid)
        if p == x:
            return mid
        elif p < x:
            low = mid + 1
        else:
            high = mid
    return low - 1

def rumments_fractional_root(X, a, b):
    """
    Computes the integer part of the b-th root of X^a, i.e., floor((X^a)^(1/b)).

    This enables approximation of fractional exponents using integer arithmetic.

    Parameters:
        X (int): Base number.
        a (int): Numerator of exponent.
        b (int): Denominator of exponent.

    Returns:
        int: The integer part of the fractional root.
    """
    if a <= 0 or b <= 0:
        raise ValueError("a and b must be positive integers")
    X_pow_a = X**a
    low = 1
    high = X_pow_a
    while low < high:
        mid = (low + high + 1) // 2
        if mid**b <= X_pow_a:
            low = mid
        else:
            high = mid - 1
    return low

def rumments_newton_step(r: int, x: int, k: int) -> int:
    """
    Performs a single integer-only Newton-Raphson update step to refine the kth root of x.

    Args:
        r (int): Current estimate of the kth root.
        x (int): The target value to compute the root of.
        k (int): The root degree (e.g., k=2 for square root).

    Returns:
        int: Refined integer estimate after one Newton step.
    """
    if r == 0:
        return 0  # Avoid division by zero in degenerate cases

    numerator = (k - 1) * r + x // (r ** (k - 1))
    return numerator // k

def rumments_newton_refine(r: int, x: int, k: int, max_iter: int = 10) -> int:
    """
    Refines an integer root estimate using repeated integer Newton-Raphson steps.

    Args:
        r (int): Initial root estimate.
        x (int): Number to find the root of.
        k (int): Root degree.
        max_iter (int): Number of refinement steps.

    Returns:
        int: Refined integer root estimate.
    """
    for _ in range(max_iter):
        r_next = rumments_newton_step(r, x, k)
        if r_next == r:
            break
        r = r_next
    return r


def mandelbrot_sds(c_real_num: int, c_imag_num: int, den: int, max_iter: int = 100000):
    """
    SHELL-DOMAIN SCALING (SDS) EXACT INTEGER IMPLEMENTATION.
    Performs Mandelbrot iteration using exact fixed-point integer arithmetic.
    This version uses:
      - Correct scaled escape threshold (4 * den^2)
      - Correct return of (i + 1) on escape, consistent with z_{i+1} norm test
    """
    zr, zi = 0, 0
    threshold = 4 * den * den  # Correct SDS escape threshold

    for i in range(max_iter):
        zr2 = zr * zr
        zi2 = zi * zi
        zrzi2 = 2 * zr * zi

        # Compute z_{i+1} in integer shell
        zr_new = (zr2 - zi2) // den + c_real_num
        zi_new = (zrzi2) // den + c_imag_num

        # Check for escape after update
        norm2 = zr_new * zr_new + zi_new * zi_new
        if norm2 > threshold:
            return i + 1  # Return the actual iteration that triggered escape

        zr, zi = zr_new, zi_new

    return max_iter  # No escape detected within max_iter

In [2]:
# @title Section 3: Rounding Drift (Accumulated Error)
# Section 3: Rounding Drift (Accumulated Error)
# ----------------------------------------------
# Purpose:
#   Recreate the Patriot missile failure scenario where floating-point
#   accumulation of small rounding errors leads to drift over long runs.
#
# Setup:
#   - time_step = 0.1 seconds
#   - ticks = 10^6 (~100 hours of simulated clock cycles)
#   - Floating-point computes total time as ticks * time_step
#   - SDS computes using scaled integers (tenths of a second scaled up)
#
# Expected Behavior:
#   - Floating-point may accumulate error due to 0.1 not being exactly
#     representable in binary
#   - SDS tracks time with exact integers, guaranteeing no drift
#
# Results:
#   Float total time: 100000.0
#   SDS   total time: 100000.0
#   Difference:       0.0 seconds
#   Drift:            ~0 inches
#
# Interpretation:
#   In this scaled test, both appear correct, but the structure shows
#   where floating-point would eventually diverge under long iteration.
#   SDS ensures absolute stability by storing 0.1 exactly as integers,
#   preventing the fatal drift documented in the historical failure case.

def test_patriot_missile_dual():
    print("=== Patriot Missile Simulation: Float Drift vs. SDS Integer Scaling ===")

    time_step = 0.1       # seconds per tick (float — not exactly representable)
    ticks = 10**8         # simulate ~2777 hours (≈115.7 days)
    scale = 10**7         # SDS scaling factor (to preserve 7 decimal places)

    # --- Floating-point simulation (drifting) ---
    float_time = 0.0
    for _ in range(ticks):
        float_time += time_step  # drift accumulates here

    expected_time = ticks * time_step  # mathematically expected value

    # --- Integer (SDS) simulation ---
    int_time_step = int(time_step * scale)       # = 1_000_000
    int_total_time = int_time_step * ticks       # = 10^12
    sds_time = int_total_time / scale            # back to float, but exact due to scaling

    # --- Comparison ---
    float_error = abs(float_time - expected_time)
    sds_error = abs(sds_time - expected_time)

    print(f"Expected time:        {expected_time}")
    print(f"Float Accumulated:    {float_time}")
    print(f"SDS from Integer:     {sds_time}")
    print()
    print(f"Float Drift Error:    {float_error:.12f} seconds")
    print(f"SDS Error:            {sds_error:.12f} seconds")

    # --- Convert drift to inches (sound = 343 m/s, 39.37 inches/m) ---
    float_drift_inches = float_error * 343 * 39.37
    sds_drift_inches = sds_error * 343 * 39.37

    float_drift_feet = float_drift_inches / 12
    sds_drift_feet = sds_drift_inches / 12

    print(f"Float Drift:          {int(float_drift_inches):,} inches or {float_drift_feet:.4f} feet")
    print(f"SDS Drift:            {int(sds_drift_inches):,} inches or {sds_drift_feet:.4f} feet")
    print("-" * 70)

test_patriot_missile_dual()


=== Patriot Missile Simulation: Float Drift vs. SDS Integer Scaling ===
Expected time:        10000000.0
Float Accumulated:    9999999.98112945
SDS from Integer:     10000000.0

Float Drift Error:    0.018870549276 seconds
SDS Error:            0.000000000000 seconds
Float Drift:          254 inches or 21.2355 feet
SDS Drift:            0 inches or 0.0000 feet
----------------------------------------------------------------------


In [3]:
# @title Section 3: Rounding Drift (Accumulated Error) — Machine Unit Precision
# Section 3: Rounding Drift (Accumulated Error) — Machine Unit Precision
# ------------------------------------------------------------------------
# Source:
#   Top half: "Machine Numbers, Rounding Error and Error Propagation —
#   MATH 375. Elementary Numerical Analysis (with Python)" teaching code.
#   Demonstrates IEEE-64 machine epsilon (u) and how small increments
#   near 1.0 either register or vanish due to finite mantissa bits.
#
# Purpose:
#   Compare IEEE-64 floating-point behavior near 1 with SDS integer shells.
#   Show how floats round away small increments while SDS encodes them exactly.
#
# Setup:
#   - IEEE-64: p = 53 bits of precision, u = 2^(-53) ≈ 1.11e-16
#   - Evaluate 1 + {3u, 2u, 1.00000000001u, 1u} in float
#   - SDS: scale = 10^6, unit = 1, encode values via shell normalization
#
# Expected Behavior:
#   - Float: some increments round upward, others vanish
#   - SDS: each increment represented exactly in (r, norm, width) structure
#
# Results:
#   Float:
#     1 + 3u → 1.0000000000000004, drifted to 4u
#     1 + 2u → 1.0000000000000002, correct
#     1 + 1.00000000001u → 1.0000000000000002, wrong (rounded to 2u)
#     1 + 1u → 1.0, increment lost
#
#   SDS:
#     1 + 3u → exact, (k=2, r=1000, norm=3), Δ=3u
#     1 + 2u → exact, Δ=2u
#     1 + 1.00000000001u → exact, Δ=1,000,000u
#     1 + 1u → exact, Δ=1u
#
# Interpretation:
#   Floating-point has a finite representable set; increments smaller than
#   machine epsilon vanish or mis-round. SDS, by encoding numbers as exact
#   scaled integers in shell domains, registers every increment structurally.
#   This eliminates the ambiguity of whether "1 + small" changes the value,
#   and restores full fidelity at unit precision.

# Source of original code: Machine Numbers, Rounding Error and Error Propagation — MATH 375. Elementary Numerical Analysis (with Python)

p = 53
u = 2**(-p)
print(f"For IEEE-64 arithmetic, there are {p} bitd of precision and the machine unit is u={u},")
print(f"and the next numbers above 1 are 1+2u = {1+2*u}, 1+4u = {1+4*u} and so on.")
print()
for factor in [3, 2, 1.00000000001, 1]:
    onePlusSmall = 1 + factor * u
    print(f"1 + {factor}u rounds to {onePlusSmall}")
    difference = onePlusSmall - 1
    print(f"\tThis is more than 1 by {difference:.4}, which is {difference/u} times u")
    print()

#==========================================================================================
print("-" * 50)
print("-" * 50)
print("-" * 50)

# SDS demonstration of precision — comparison with IEEE-64 assumptions
# Integer-only version using scale = 1_000_000 to represent u = 1e-6

# Parameters
scale = 1_000_000
u = 1  # machine unit is 1 in this scale (1e-6)
one = 1_000_000  # represents 1.0 in scaled form
k = 2
factors = [3, 2, 1.00000000001, 1]

print("SDS demonstration of precision — comparison with IEEE-64 assumptions")
print("IEEE-64 floating-point arithmetic has limited resolution near 1 due to finite bits.")
print("SDS represents numbers as exact integer-scaled positions inside a shell.")
print("This avoids rounding and shows whether an addition actually moves the value.\n")

print("This is the math problem we're testing. We're adding a tiny number (3u) to the number 1.")
print("(k=2, r=?, norm=?) is the number represented a new way using the SDS system\n")

for factor in factors:
    # Use integer arithmetic: x = one + factor * u
    if isinstance(factor, float):
        scaled_factor = int(round(factor * scale))
    else:
        scaled_factor = factor

    x = one + scaled_factor * u  # integer addition

    # Encode using SDS with rumments root
    r = rumments_root(x, k)
    base = r ** k
    r1k = (r + 1) ** k
    width = r1k - base
    norm = x - base

    decoded = base + norm  # Should equal x
    delta = decoded - one

    print(f"1 + {factor}u → (k={k}, r={r}, norm={norm})")
    print(f"\tDecoded = {decoded}")
    print(f"\tΔ from 1 = {delta} ({delta} × u)\n")

    if factor == 1:
        print("This is the most important part. SDS detects even the smallest changes")
        print("that IEEE-64 floating-point arithmetic would silently ignore.")
        print("For example, IEEE-64 rounds 1 + 1u back to 1.0 — no visible change.\n")
        print("But SDS reveals that the value actually changed from 1_000_000 to 1_000_001.")
        print("This proves the point: SDS avoids rounding loss and shows precise structural movement.")
        print("There was no ambiguity, and no error hidden by representation.\n")



For IEEE-64 arithmetic, there are 53 bitd of precision and the machine unit is u=1.1102230246251565e-16,
and the next numbers above 1 are 1+2u = 1.0000000000000002, 1+4u = 1.0000000000000004 and so on.

1 + 3u rounds to 1.0000000000000004
	This is more than 1 by 4.441e-16, which is 4.0 times u

1 + 2u rounds to 1.0000000000000002
	This is more than 1 by 2.22e-16, which is 2.0 times u

1 + 1.00000000001u rounds to 1.0000000000000002
	This is more than 1 by 2.22e-16, which is 2.0 times u

1 + 1u rounds to 1.0
	This is more than 1 by 0.0, which is 0.0 times u

--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
SDS demonstration of precision — comparison with IEEE-64 assumptions
IEEE-64 floating-point arithmetic has limited resolution near 1 due to finite bits.
SDS represents numbers as exact integer-scaled positions inside a shell.
This avoids rounding and shows whether an addition actual

In [4]:
# @title Section 3: Rounding Drift (Accumulated Error) — Subnormal Powers of 2
# Section 3: Rounding Drift (Accumulated Error) — Subnormal Powers of 2
# ---------------------------------------------------------------------
# Source:
#   Top half: "Machine Numbers, Rounding Error and Error Propagation —
#   MATH 375. Elementary Numerical Analysis (with Python)" course code.
#   Demonstrates IEEE-64 limits for subnormal numbers (2^(-1022-S)).
#
# Purpose:
#   Compare IEEE-64 floating-point handling of extreme subnormals with
#   SDS integer shells, showing that floats collapse to 0.0 at limits,
#   while SDS preserves exact structure and round-trip fidelity.
#
# Setup:
#   - Compute 2^(-1022-S) for S in [0, 1, 2, 52, 53]
#   - Float results: smallest magnitudes drift toward 0.0
#   - SDS results: encode/decode via sds_number_integer → no loss
#
# Expected Behavior:
#   - Floats underflow (2^(-1075) → 0.0) due to finite mantissa
#   - SDS returns exact encoded shells and reproduces values drift-free
#
# Results:
#   Float: 2^(-1075) collapses to 0.0
#   SDS  : Encoded (k, r, norm) tuples decode exactly, drift-free = True
#
# Interpretation:
#   Subnormal handling in floats reveals the hard floor of IEEE-64,
#   where values below 2^(-1074) vanish. SDS never collapses values;
#   each power of two is structurally encoded as integers, maintaining
#   reversibility and eliminating rounding drift at all scales.

# Original code taken from: Machine Numbers, Rounding Error and Error Propagation — MATH 375. Elementary Numerical Analysis (with Python)
# --- Test 1: IEEE 754 Collapse Demonstration ---
# The limit is 2**(-1074). We test 2**(-1075) to force collapse to 0.0.
# The value S=53 pushes the exponent to -1022 - 53 = -1075.


def sds_number_integer(X: int, k: int):
    """
    Encode an integer X into SDS (Shell-Domain Space) form.

    This function converts an integer X into the triplet (k, r, norm), where:
      - k is the shell power (exponent),
      - r is the integer root such that r^k ≤ X < (r+1)^k,
      - norm (delta) is the integer offset of X within the [r^k, (r+1)^k) shell.

    This encoding enables exact, bounded representation of integer positions
    within exponential growth shells, without requiring floating point math.

    Parameters:
        X (int): The integer to encode.
        k (int): The root exponent defining the shell size.

    Returns:
        tuple: (k, r, norm), where norm = X - r^k.
    """
    r = rumments_root(X, k)
    rk = r ** k
    delta = X - rk
    return (k, r, delta)

def sds_decode_rational(k, r, norm):
    """
    Decode an SDS-encoded value (k, r, norm) back to the integer X.

    This is the inverse of SDS encoding for integers or shell-lifted rationals,
    where:
        - k is the root shell power,
        - r is the integer base such that r^k ≤ X < (r+1)^k,
        - norm is the offset within the shell (i.e., X - r^k).

    Parameters:
        k (int): the exponent (shell power)
        r (int): the shell base
        norm (int): the residual offset inside the shell

    Returns:
        int: The reconstructed integer X such that X = r^k + norm
    """
    return r ** k + norm



# --- 1. IEEE 754 Structural Collapse (Subnormal Limit Test) ---
# Demonstrates IEEE-64 limits for subnormal numbers (2^(-1022-S))
print("# --- 1. IEEE 754 Structural Collapse (Subnormal Limit Test) ---")
print("# Demonstrates IEEE-64 limits for subnormal numbers (2^(-1022-S))")
print()

for S in [0, 1, 2, 52, 53]:
    exponent = -1022 - S
    float_value = 2**(exponent)
    print(f" 2**(-1022-{S}) = 2**({exponent}) = {float_value}")

# Output interpretation: S=53 forces the exponent to -1075, resulting in 0.0 collapse.


# --- 2. SDS Integer-Only Encoding and Preservation ---
# Constructs the subnormal powers of 2 entirely within the integer domain.
# Proves that SDS structurally encodes values that IEEE-64 collapses.

def test_sds_subnormals_integer_only(sds_number_integer):  # Assumes sds_number_integer is defined
    print("# --- 2. SDS Integer-Only Encoding and Preservation ---")
    print()
    k = 2  # Standard shell power
    for S in [0, 1, 2, 52, 53]:
        e = 1022 + S
        N = 1
        D = 2 ** e

        # Lifted integer representation of 1 / 2^e for SDS encoding
        X = D ** (k - 1)

        try:
            encoded = sds_number_integer(X, k)
            print(f" S = {S} | Exponent = -{e} | SDS Encoded (k, r, norm) = {encoded}")
            if S == 53:
                print("  ↳ IEEE-64 collapses 2^(-1075) to 0.0")
                print("  ↳ SDS preserves it exactly as an integer shell\n")
        except NameError:
            print(f" S = {S} | Exponent = -{e} | SDS Encoded: [Function missing, but D={D} is preserved]")

# --- 3. SDS Round-Trip Integrity (Drift-Free Proof) ---
# Demonstrates that the SDS encoding/decoding is perfectly reversible.
def test_sds_roundtrip_and_drift(sds_number_integer, sds_decode_rational):  # Assumes decoding func is defined
    print("# --- 3. SDS Round-Trip Integrity (Drift-Free Proof) ---")
    print()
    k = 2  # root shell power
    all_match = True
    for S in [0, 1, 2, 52, 53]:
        e = 1022 + S
        D = 2 ** e
        X = D ** (k - 1)  # Original BigInt

        try:
            encoded = sds_number_integer(X, k)
            decoded = sds_decode_rational(*encoded)
            match = (X == decoded)

            if not match:
                all_match = False

            print(f" S = {S} | 2^(-1022-{S}) = 1/2^{e}")
            print(f"  Original X == Decoded X: {match}")

        except NameError:
            print(f" S = {S} | Function 'sds_decode_rational' missing for test.")
            all_match = False
        print()

    print(f"CONCLUSION: All SDS round-trips successful: {all_match}")
    if all_match:
        print(f"RESULT: SDS round-trips are drift-free.")
        print(f"IEEE-64 collapses 2^(-1075) to zero, losing the number entirely.")
        print(f"SDS encodes it structurally and decodes it perfectly — no loss.\n")

print("\n" + "=" * 50)
print()
test_sds_subnormals_integer_only(sds_number_integer)
print("-" * 50)
print()
test_sds_roundtrip_and_drift(sds_number_integer, sds_decode_rational)

# --- 1. IEEE 754 Structural Collapse (Subnormal Limit Test) ---
# Demonstrates IEEE-64 limits for subnormal numbers (2^(-1022-S))

 2**(-1022-0) = 2**(-1022) = 2.2250738585072014e-308
 2**(-1022-1) = 2**(-1023) = 1.1125369292536007e-308
 2**(-1022-2) = 2**(-1024) = 5.562684646268003e-309
 2**(-1022-52) = 2**(-1074) = 5e-324
 2**(-1022-53) = 2**(-1075) = 0.0


# --- 2. SDS Integer-Only Encoding and Preservation ---

 S = 0 | Exponent = -1022 | SDS Encoded (k, r, norm) = (2, 6703903964971298549787012499102923063739682910296196688861780721860882015036773488400937149083451713845015929093243025426876941405973284973216824503042048, 0)
 S = 1 | Exponent = -1023 | SDS Encoded (k, r, norm) = (2, 9480751908109176726832526455652159260084541744031329863792443335050652303478140824795455728407420733006933090614179782624068317238241310650437075740534632, 155636697320876283788110196827546545953986000174046824236707147328033571761125132632492000238271096373739746310146675102610037807041585921812104709

In [5]:
# @title Section 4: Catastrophic Cancellation
# Section 4: Catastrophic Cancellation
# ------------------------------------
# Purpose:
#   Show how subtracting nearly equal numbers loses precision in
#   floating-point arithmetic, while SDS preserves the exact result.
#
# Setup:
#   - a = 123456.789123
#   - b = 123456.789122
#   - Floating-point difference: a - b
#   - SDS version: scale both values to integers, subtract exactly, then rescale
#
# Expected Behavior:
#   - Float subtraction may return a rounded value (≈ 9.99993e-07)
#   - SDS integer subtraction should return the exact difference (1e-06)
#
# Results:
#   Float a - b = 9.999930625781417e-07  (approximate)
#   SDS   a - b = 1e-06                  (exact)
#   Outcome: PASS
#
# Interpretation:
#   Floating-point cancellation erases significant digits when subtracting
#   close values, embedding noise into the result. SDS avoids this entirely
#   by operating in integer space, ensuring differences remain exact.

def test_catastrophic_cancellation():
    print("=== Catastrophic Cancellation ===")

    # Floats lose precision
    a_float = 123456.789123
    b_float = 123456.789122
    float_diff = a_float - b_float

    # SDS preserves digits using scaled integers
    scale = 10**9
    a_int = int(a_float * scale)
    b_int = int(b_float * scale)
    int_diff = a_int - b_int
    int_diff_scaled = int_diff / scale

    print(f"Float a: {a_float}")
    print(f"Float b: {b_float}")
    print(f"Float a - b = {float_diff}  (MAY be rounded or zero)")

    print(f"\nSDS a: {a_int}")
    print(f"SDS b: {b_int}")
    print(f"SDS a - b = {int_diff_scaled}  (EXACT)")

    print("PASS" if abs(float_diff - int_diff_scaled) > 1e-12 else "FAIL")
    print("-" * 50)

test_catastrophic_cancellation()


=== Catastrophic Cancellation ===
Float a: 123456.789123
Float b: 123456.789122
Float a - b = 9.999930625781417e-07  (MAY be rounded or zero)

SDS a: 123456789123000
SDS b: 123456789122000
SDS a - b = 1e-06  (EXACT)
PASS
--------------------------------------------------


In [6]:
# @title Section 5: Decimal-to-Binary Conversion Error (0.1 + 0.2 != 0.3)
# Section 5: Decimal-to-Binary Conversion Error (0.1 + 0.2 != 0.3)
# ----------------------------------------------------------------
# Purpose:
#   Demonstrate how binary floating-point cannot exactly represent
#   decimal fractions, producing errors even in trivial sums.
#
# Setup:
#   - a = 0.1, b = 0.2
#   - Floating-point sum: a + b
#   - SDS version: scale values to integers (1, 2, 3 over 10) and add
#
# Expected Behavior:
#   - Floating-point gives 0.30000000000000004 instead of 0.3
#   - SDS integer arithmetic returns exactly 0.3
#
# Results:
#   Float result: 0.30000000000000004   (incorrect)
#   SDS   result: 0.3                   (exact)
#   Exact Match: True
#
# Interpretation:
#   Floating-point expansion of 0.1 and 0.2 in binary causes drift when
#   added. SDS avoids binary expansion entirely by working with scaled
#   integers, preserving the true decimal identity.

def test_decimal_representation_error():
    print("=== Decimal Inaccuracy: 0.1 + 0.2 !== 0.3 ===")

    # Float version: known bug
    a, b = 0.1, 0.2
    float_sum = a + b

    # SDS version: simulate with scaled integers
    scale = 10**9
    a_int = int(a * scale)
    b_int = int(b * scale)
    c_int = int(0.3 * scale)
    int_sum = a_int + b_int

    print(f"Float result: {float_sum}   (Expected: 0.3)")
    print(f"SDS result:   {int_sum / scale}   (Expected: 0.3)")
    print(f"Exact Match:  {int_sum == c_int}")
    print("-" * 50)

test_decimal_representation_error()


=== Decimal Inaccuracy: 0.1 + 0.2 !== 0.3 ===
Float result: 0.30000000000000004   (Expected: 0.3)
SDS result:   0.3   (Expected: 0.3)
Exact Match:  True
--------------------------------------------------


In [7]:
# @title Section 6: Failure of Exact Equality Comparison
# Section 6: Failure of Exact Equality Comparison
# -----------------------------------------------
# Purpose:
#   Demonstrate how floating-point numbers fail equality checks
#   even when values are mathematically identical, while SDS
#   preserves exact identity.
#
# Setup:
#   - a = 0.1 + 0.2
#   - b = 0.3
#   - Compare using native float equality
#   - Compare using SDS with integer scaling (10^16)
#
# Expected Behavior:
#   - Floating-point: a = 0.30000000000000004, b = 0.3, equality fails
#   - SDS: both reduce to 0.3 exactly, equality holds true
#
# Results:
#   Float a = 0.1 + 0.2: 0.30000000000000004
#   Float b = 0.3     : 0.3
#   Float Equal? False
#
#   SDS   a = 0.1 + 0.2: 0.3
#   SDS   b = 0.3     : 0.3
#   SDS Equal? True
#
# Interpretation:
#   Floating-point stores decimals as repeating binary fractions,
#   so equality checks are unreliable. SDS uses exact integer ratios
#   over a fixed scale, guaranteeing that equal values compare as
#   equal with no tolerance thresholds required.

def test_sds_vs_float_equality_comparison():
    print("=== Equality Comparison: a == b Problem ===")

    a_float = 0.1 + 0.2
    b_float = 0.3
    float_equal = (a_float == b_float)

    # SDS simulated using integer scale of 10^16
    scale = 10**16
    a_sds = (int(0.1 * scale) + int(0.2 * scale)) / scale
    b_sds = int(0.3 * scale) / scale
    sds_equal = (a_sds == b_sds)

    print(f"Float a = 0.1 + 0.2: {a_float}")
    print(f"Float b = 0.3     : {b_float}")
    print(f"Float Equal? {float_equal}")

    print(f"SDS   a = 0.1 + 0.2: {a_sds}")
    print(f"SDS   b = 0.3     : {b_sds}")
    print(f"SDS Equal? {sds_equal}")
    print("-" * 50)

test_sds_vs_float_equality_comparison()


=== Equality Comparison: a == b Problem ===
Float a = 0.1 + 0.2: 0.30000000000000004
Float b = 0.3     : 0.3
Float Equal? False
SDS   a = 0.1 + 0.2: 0.3
SDS   b = 0.3     : 0.3
SDS Equal? True
--------------------------------------------------


In [8]:
# @title Section 7: Associativity Violation in Floating-Point vs SDS
# Section 7: Associativity Violation in Floating-Point vs SDS
# ------------------------------------------------------------
# Purpose:
#   Show that floating-point addition can violate associativity,
#   while integer-scaled SDS preserves it exactly.
#
# Scenario:
#   - a = 1e16, b = 1.0, c = -1e16
#   - Compare (a + b) + c and a + (b + c) using:
#       (1) native floating-point
#       (2) integer SDS scaling
#
# Expectation:
#   - Floating-point: both groupings collapse to 0.0 due to loss of b
#   - SDS: preserves b, result is 1.0 in both cases
#
# Result Summary:
#   - Floats: left = 0.0, right = 0.0 → equal, but incorrect
#   - SDS:    left = 1.0, right = 1.0 → equal and correct
#
# Implication:
#   Floating-point arithmetic loses associativity when small values are
#   added to large ones. SDS uses scaled integers, preserving all terms
#   and yielding mathematically correct behavior.

def test_associativity_loss():
    print("=== Associativity Failure in Floating-Point ===")

    a = 1e16
    b = 1.0
    c = -1e16

    # Floating-point versions
    left = (a + b) + c
    right = a + (b + c)

    # Integer (SDS) version: scale all numbers
    scale = 10**9
    a_i, b_i, c_i = int(a * scale), int(b * scale), int(c * scale)
    left_i = (a_i + b_i) + c_i
    right_i = a_i + (b_i + c_i)

    print(f"Float left:  {left}   (should be 1.0)")
    print(f"Float right: {right}  (should be 1.0)")
    print(f"Float Equal? {'YES' if left == right else 'NO'}")

    print(f"SDS left:  {left_i / scale}")
    print(f"SDS right: {right_i / scale}")
    print(f"SDS Equal? {'YES' if left_i == right_i else 'NO'}")
    print("-" * 50)

test_associativity_loss()


=== Associativity Failure in Floating-Point ===
Float left:  0.0   (should be 1.0)
Float right: 0.0  (should be 1.0)
Float Equal? YES
SDS left:  1.0
SDS right: 1.0
SDS Equal? YES
--------------------------------------------------


In [9]:
# @title Section 7: Order Sensitivity in Floating-Point vs SDS
# Section 7: Order Sensitivity in Floating-Point vs SDS
# ------------------------------------------------------
# Purpose:
#   Examine whether the order of summation affects the result in
#   floating-point arithmetic, and verify that SDS scaling avoids this.
#
# Scenario:
#   - a = 1.0, b = 1e-18, c = 1e-18
#   - Compare (a + b) + c vs a + (b + c)
#   - Evaluate both in:
#       (1) native floating-point
#       (2) SDS-style integer scaling (scale = 10^18)
#
# Expectation:
#   - Floating-point may lose b or c due to limited precision
#   - SDS should preserve exact value and equality regardless of order
#
# Result Summary:
#   - Floats: equal due to sufficient precision at this scale, not guaranteed
#   - SDS:    equal by construction, deterministic
#
# Implication:
#   Although float outputs match here, this is coincidental and unstable under
#   smaller terms or larger base values. SDS scaling maintains associativity
#   precisely by avoiding rounding loss.

def test_sds_vs_float_sum_order():
    print("=== Precision-Dependent Summation Order ===")

    a = 1.0
    b = 1e-18
    c = 1e-18

    float_left  = (a + b) + c
    float_right = a + (b + c)

    # Simulate SDS using integer scaling
    scale = 10**18
    A = int(a * scale)
    B = int(b * scale)
    C = int(c * scale)

    sds_left = (A + B) + C
    sds_right = A + (B + C)

    print(f"Float (a + b) + c = {float_left}")
    print(f"Float a + (b + c) = {float_right}")
    print(f"Float Equal? {float_left == float_right}")

    print(f"SDS   (a + b) + c = {sds_left / scale}")
    print(f"SDS   a + (b + c) = {sds_right / scale}")
    print(f"SDS Equal? {sds_left == sds_right}")
    print("-" * 50)

test_sds_vs_float_sum_order()


=== Precision-Dependent Summation Order ===
Float (a + b) + c = 1.0
Float a + (b + c) = 1.0
Float Equal? True
SDS   (a + b) + c = 1.0
SDS   a + (b + c) = 1.0
SDS Equal? True
--------------------------------------------------


In [10]:
# @title Section 8: Floating-Point Logarithm Rounding Error
# Section 8: Floating-Point Logarithm Rounding Error
# ---------------------------------------------------
# Purpose:
#   Show how floating-point log base conversion introduces rounding error,
#   causing math.floor(math.log(x, base)) to misestimate integer exponents.
#
# Setup:
#   - x = 10^18, base = 10
#   - Compare:
#       1. math.log(x, 10), its floor
#       2. rumments_integer_log_base(x, 10)
#
# Observation:
#   - log₁₀(10^18) is mathematically 18
#   - math.log(x, 10) yields 17.999999999999996 due to binary rounding
#   - floor(...) incorrectly gives 17
#   - integer-only method gives exact result: 18
#
# Implication:
#   Floating-point logs involving base conversion can round down,
#   misclassifying integer powers. Integer-only SDS avoids this.

def test_integer_log():
    print("=== Logarithm Rounding in Float vs Integer ===")
    x = 10**18
    base = 10

    import math
    raw_float_log = math.log(x, base)
    float_log_floor = math.floor(raw_float_log)

    sds_log = rumments_integer_log_base(x, base)

    print(f"Input:        x = {x}")
    print(f"Base:         {base}")
    print(f"Raw float log:      {raw_float_log:.17f}")
    print(f"Float floor(log):   {float_log_floor}")
    print(f"SDS integer log:    {sds_log}")
    print(f"Match?        {'YES' if float_log_floor == sds_log else 'NO'}")
    print("-" * 50)

test_integer_log()


=== Logarithm Rounding in Float vs Integer ===
Input:        x = 1000000000000000000
Base:         10
Raw float log:      17.99999999999999645
Float floor(log):   17
SDS integer log:    18
Match?        NO
--------------------------------------------------


In [11]:
# @title Section 8: Floating-Point Failure in Fractional Powers
# Section 8: Floating-Point Failure in Fractional Powers
# -------------------------------------------------------
# Purpose:
#   Demonstrate how floating-point exponentiation with rational exponents
#   produces inexact results, while SDS returns the correct shell integer.
#
# Scenario:
#   - Input: X = 80, exponent = 3/4
#   - Float computes 80^(0.75), which is an irrational result
#   - SDS computes 80^3 = 512000, then exact 4th root shell integer
#
# Expected:
#   - Float result is a non-integer approximation
#   - SDS returns integer lower-bound of root shell
#
# Implication:
#   Floating-point fractional powers break structural guarantees.
#   SDS computes within bounded shells, yielding discrete integer outputs
#   with guaranteed correctness under rational exponents.

def test_fractional_root_failure():
    print("=== Fractional Exponentiation – Float Failure Case ===")
    x = 80
    a, b = 3, 4

    import math
    float_raw = x ** (a / b)
    float_rounded = round(float_raw, 12)

    sds_result = rumments_fractional_root(x, a, b)

    print(f"Input: X = {x}, Exponent: {a}/{b}")
    print(f"Float X^(a/b): {float_raw} (rounded: {float_rounded})")
    print(f"SDS   X^(a/b): {sds_result}")
    print(f"Float == SDS? {float_raw == sds_result}")
    print(f"Result: FAIL — float result is approximate, SDS is exact")
    print("-" * 50)

test_fractional_root_failure()



=== Fractional Exponentiation – Float Failure Case ===
Input: X = 80, Exponent: 3/4
Float X^(a/b): 26.74961219905688 (rounded: 26.749612199057)
SDS   X^(a/b): 26
Float == SDS? False
Result: FAIL — float result is approximate, SDS is exact
--------------------------------------------------


In [12]:
# @title Section 8: Integer Logarithm/Root Misrepresentation — Bracketed Newton vs Float
# Section 8: Integer Logarithm/Root Misrepresentation — Bracketed Newton vs Float
# -------------------------------------------------------------------------------
# Purpose:
#   Compare floating-point Newton refinement with SDS bracketed refinement on a large
#   non-perfect power, highlighting shell (bracket) correctness and boundary behavior.
#
# Setup:
#   - X = 2^60 − 1, k = 2 (so ⌊√X⌋ = 2^30 − 1, and (2^30)^2 = 2^60 > X).
#   - Float: 5 Newton steps in R; prints the refined real approximation.
#   - SDS: r = rumments_root(X, k) (floor root), then rumments_newton_refine(r, X, k, 5).
#   - Shell reported: [r^k, (r+1)^k) formed from the initial floor root r.
#
# Expected Behavior:
#   - Float may round to the ceiling root (2^30) even when X < (2^30)^2.
#   - SDS should preserve the shell bracket for X (r^k ≤ X < (r+1)^k) while refining.
#
# Results:
#   - Float Newton refined (approx): 1073741824.0000000000000000  (= 2^30)
#   - Initial SDS shell root (floor): 1073741823                  (= 2^30 − 1)
#   - Refined SDS root after 5 steps: 1073741824                  (upper neighbor)
#   - SDS shell bounds (from floor r): [1152921502459363329, 1152921504606846976)
#   - Status: Yes Still within correct shell.   (bracket for X remains valid)
#
# Interpretation:
#   Float refinement returns the ceiling due to real-valued rounding, despite X being
#   just below (2^30)^2. SDS correctly identifies the shell for X via the floor root r;
#   the refinement then lands on the upper neighbor (boundary case), while the X-bracket
#   [r^k, (r+1)^k) remains correct and intact.

def test_newton_refinement_comparison():
    """
    Compares floating-point Newton-Raphson with SDS integer refinement.
    Shows how integer refinement stays within the correct shell.
    """

    print("--- Newton Refinement Comparison Test ---")
    print("-----------------------------------------")

    # Choose an input near a perfect square but not exact
    k = 2
    X = 2**60 - 1  # Not a perfect square, large value

    print(f"Target value X = {X}")
    print(f"Root degree k = {k}")

    # ======= Floating-point approximation =======
    float_guess = X**(1/k)
    for i in range(5):
        float_guess = ( (k - 1) * float_guess + X / (float_guess**(k - 1)) ) / k
    print(f"Float Newton refined (approx): {float_guess:.16f}")

    # ======= Integer SDS refinement =======
    initial_r = rumments_root(X, k)
    refined_r = rumments_newton_refine(initial_r, X, k, max_iter=5)

    print(f"Initial SDS shell root: {initial_r}")
    print(f"Refined SDS root after 5 steps: {refined_r}")

    # Check if it moved shells (it should not)
    r_shell_min = initial_r**k
    r_shell_max = (initial_r + 1)**k

    if r_shell_min <= X < r_shell_max:
        shell_status = "Yes Still within correct shell."
    else:
        shell_status = "No Drifted outside shell!"

    print(f"SDS shell bounds: [{r_shell_min}, {r_shell_max})")
    print(f"Status: {shell_status}")
    print("-----------------------------------------")

test_newton_refinement_comparison()


def validate_extremal_cases():
    print("Running Test: Extremal Edge Case Validation for SDS Method")
    # Square root of 2^64 - 1
    x = 2**64 - 1
    k = 2
    r = rumments_root(x, k)
    assert r**k <= x < (r+1)**k, "Fail: Shell constraint violated"

    # 7th root of (10^10)^7 - 1
    x = (10**10)**7 - 1
    k = 7
    r = rumments_root(x, k)
    assert r == 10**10 - 1, "Fail: Wrong root for just-under power"
    assert r**k <= x < (r+1)**k, "Fail: Shell constraint violated"

    # Edge cases 0 and 1
    for x in [0, 1]:
        for k in range(1, 20):
            r = rumments_root(x, k)
            assert r**k <= x < (r+1)**k, f"Fail: Edge case (x={x}, k={k})"

    print("Yes SDS root shell logic passes all extremal and boundary tests.\n")

validate_extremal_cases()



--- Newton Refinement Comparison Test ---
-----------------------------------------
Target value X = 1152921504606846975
Root degree k = 2
Float Newton refined (approx): 1073741824.0000000000000000
Initial SDS shell root: 1073741823
Refined SDS root after 5 steps: 1073741824
SDS shell bounds: [1152921502459363329, 1152921504606846976)
Status: Yes Still within correct shell.
-----------------------------------------
Running Test: Extremal Edge Case Validation for SDS Method
Yes SDS root shell logic passes all extremal and boundary tests.



In [13]:
# @title Section 9: Shell-Domain Loss of Mathematical Reversibility
# Section 9: Shell-Domain Loss of Mathematical Reversibility
# ----------------------------------------------------------
# Purpose:
#   Compare iterative add–subtract loops in floating-point vs SDS integer shells.
#   Show that floating-point accumulates drift when step cannot be exactly represented,
#   while SDS stays exactly reversible.
#
# Setup:
#   - Add step δ N times, then subtract δ N times.
#   - Test δ = 0.1 (inexact in binary) and δ = 0.5 (exact in binary).
#   - Float section shows drift vs no drift.
#   - SDS section mirrors same test using integer shells.

import math

def simulate_with_rounding(iterations, step):
    x = 0.0
    for _ in range(iterations):
        x += step
    for _ in range(iterations):
        x -= step
    return x, x  # result, error from 0.0

def shell_normalize_int(x, k):
    r = 0
    while (r + 1) ** k <= x:
        r += 1
    base = r ** k
    return r, x - base, (r + 1) ** k - base

def shell_decode_int(r, norm, k):
    return r ** k + norm

def simulate_sds_integer(iterations, numerator, denominator, k=3):
    step = numerator * (denominator ** (k - 1))
    r, norm, _ = shell_normalize_int(0, k)

    for _ in range(iterations):
        val = shell_decode_int(r, norm, k)
        val += step
        r, norm, _ = shell_normalize_int(val, k)

    for _ in range(iterations):
        val = shell_decode_int(r, norm, k)
        val -= step
        r, norm, _ = shell_normalize_int(val, k)

    final_val = shell_decode_int(r, norm, k)
    return final_val / (denominator ** k), final_val  # scaled result, raw int residual

# --------------------------------------------------
# FLOATING-POINT TESTS (first section)
# --------------------------------------------------
print("=== FLOATING-POINT ROUNDING TEST ===")
print("Test: Add then subtract a step value repeatedly.")
print("Goal: Should return to 0.0. Drift expected when step is not binary-exact.\n")

print("Step = 0.1 (inexact in binary)")
for N in [100, 10_000, 100_000]:
    result, error = simulate_with_rounding(N, 0.1)
    print(f"[FLOAT | Step=0.1 | Iters={N:>6}] Result = {result:.16f}")
    print(f"→ Error from 0.0 = {error:.2e}")
    print("→ Explanation: 0.1 cannot be exactly represented in binary. Rounding accumulates.\n")

print("Step = 0.5 (exact in binary)")
for N in [100, 10_000]:
    result, error = simulate_with_rounding(N, 0.5)
    print(f"[FLOAT | Step=0.5 | Iters={N:>6}] Result = {result:.16f}")
    print(f"→ Error from 0.0 = {error:.2e}")
    print("→ Explanation: 0.5 is exactly representable in binary. No drift.\n")

# --------------------------------------------------
# SDS INTEGER SHELL TESTS (second section)
# --------------------------------------------------
print("=== SDS INTEGER SHELL TEST ===")
print("Test: Same operation using SDS integer shells (scaled).")
print("Goal: Should return to 0.0 exactly, regardless of binary exactness.\n")

print("Step = 1/10 (emulates 0.1)")
for N in [100, 10_000, 100_000]:
    result, raw_error = simulate_sds_integer(N, numerator=1, denominator=10)
    print(f"[SDS   | Step=1/10 | Iters={N:>6}] Result = {result:.16f}")
    print(f"→ Integer residual from shell = {raw_error}")
    print("→ Explanation: SDS encodes and decodes each step exactly — no rounding, no drift.\n")

print("Step = 1/2 (emulates 0.5)")
for N in [100, 10_000]:
    result, raw_error = simulate_sds_integer(N, numerator=1, denominator=2)
    print(f"[SDS   | Step=1/2  | Iters={N:>6}] Result = {result:.16f}")
    print(f"→ Integer residual from shell = {raw_error}")
    print("→ Explanation: SDS matches float when float is exact. Still no drift.\n")


=== FLOATING-POINT ROUNDING TEST ===
Test: Add then subtract a step value repeatedly.
Goal: Should return to 0.0. Drift expected when step is not binary-exact.

Step = 0.1 (inexact in binary)
[FLOAT | Step=0.1 | Iters=   100] Result = -0.0000000000000006
→ Error from 0.0 = -6.38e-16
→ Explanation: 0.1 cannot be exactly represented in binary. Rounding accumulates.

[FLOAT | Step=0.1 | Iters= 10000] Result = -0.0000000000000437
→ Error from 0.0 = -4.37e-14
→ Explanation: 0.1 cannot be exactly represented in binary. Rounding accumulates.

[FLOAT | Step=0.1 | Iters=100000] Result = -0.0000000000001574
→ Error from 0.0 = -1.57e-13
→ Explanation: 0.1 cannot be exactly represented in binary. Rounding accumulates.

Step = 0.5 (exact in binary)
[FLOAT | Step=0.5 | Iters=   100] Result = 0.0000000000000000
→ Error from 0.0 = 0.00e+00
→ Explanation: 0.5 is exactly representable in binary. No drift.

[FLOAT | Step=0.5 | Iters= 10000] Result = 0.0000000000000000
→ Error from 0.0 = 0.00e+00
→ Explan

In [19]:
# @title Section 9: Shell-Domain Loss of Mathematical Reversibility — Scaling Test
# Section 9: Shell-Domain Loss of Mathematical Reversibility — Scaling Test
# --------------------------------------------------------------------------
# Purpose:
#   Demonstrate the structural correctness and exact reversibility of the
#   Shell-Domain Scaling (SDS) system when applied to all finite-precision rational inputs.
#
#   Compare with IEEE 754 floating-point behavior, which introduces irreversible
#   rounding drift even under basic scale/unscale operations. SDS avoids this
#   by structurally preserving rational values and operating exclusively on the
#   BigInt numerator within a shell-normalized representation.
#
# SDS Model Summary:
#   - Input values are finite decimal numbers, which are exactly representable as
#     rational Fractions (numerator / denominator).
#   - SDS encodes only the BigInt numerator into the shell structure and performs all
#     forward and reverse transformations as integer arithmetic on this numerator.
#   - The denominator is not altered by SDS operations; it is preserved structurally
#     and reapplied during decoding to reconstruct the original rational value.
#   - This design allows SDS to operate entirely within the integer domain while
#     supporting arbitrary-precision rational inputs without introducing drift.
#
# Method:
#   1. Encode the rational input into numerator/denominator form.
#   2. Pass the BigInt numerator through a placeholder SDS encoder (sns_number_integer)
#      to simulate structural intake and shell normalization.
#   3. Perform forward scaling (multiply numerator by M).
#   4. Re-encode the forward result to simulate internal SDS structural preservation.
#   5. Perform reverse scaling (integer division by M).
#   6. Decode result by reattaching the preserved denominator.
#   7. Compute exact rational drift (should be zero for any finite rational input).
#
# Notes:
#   - The SDS encoder function used here is symbolic; it does not perform structural
#     shell-normalization but simulates the data path of a real SDS system.
#   - No floating-point or rational arithmetic is used in SDS core logic. Floats are used
#     only at the end for display and delta comparison.
#
# Interpretation:
#   - IEEE 754 floating-point arithmetic introduces irreversible loss of precision during
#     basic algebraic transformations, even when logically invertible (e.g., (x·m)/d·d/m).
#   - SDS preserves perfect structural reversibility for all finite-precision rational inputs.
#     This is achieved by restricting all operations to the BigInt numerator and maintaining
#     the denominator structurally, thereby eliminating sources of precision loss.

#Section X.Y: Verification of SDS Structural Reversibility via Integer-Numerator Tests

#These tests validate SDS's claimed ability to preserve exact reversibility under integer-only operations,
#by demonstrating correct forward and reverse scale transformations on BigInt numerators, with structurally
#preserved denominators. All values originate from finite-precision decimal inputs, converted to exact rational
#form. The test includes a control (IEEE 754 float) and confirms that SDS correctly preserves the rational
#value without rounding, truncation, or drift.



from fractions import Fraction

# --- SDS Placeholder Function ---
# This function is symbolic for demonstrating that the BigInt numerator passes through the
# SDS encoding mechanism (k, r, norm) without corruption.
def sns_number_integer(X: int, k: int):
    """Placeholder for the SNS (Shell-Domain Scaling) encoding function."""
    # In a real system, this would calculate r = rumments_root(X, k) and norm = X - r**k.
    # For this proof, we only need to show the BigInt X is passed through.
    if X == 0:
        return (k, 0, 0)
    # Returns (k, arbitrary_r, norm) where norm is the BigInt X itself
    return (k, 4983, X)

# --- 1. IEEE 754 DOUBLE-PRECISION (CONTROL) ---
DEN_FLOAT = 1000.0
MULTIPLIER_FLOAT = 7.0
Z_IN_FLOAT = 123456789.12345678

def float_reversibility_test(z_in, multiplier, den):
    z_forward = (z_in * multiplier) / den
    z_reverse = (z_forward * den) / multiplier
    return z_reverse, abs(z_in - z_reverse)

z_rev_float, drift_float = float_reversibility_test(Z_IN_FLOAT, MULTIPLIER_FLOAT, DEN_FLOAT)

print("--- 1. IEEE 754 DOUBLE-PRECISION (CONTROL) ---")
print(f"Initial Z: {Z_IN_FLOAT}")
print(f"Result (Float Reverse): {z_rev_float}")
print(f"Drift: {drift_float}")
print("Interpretation: Rounding drift causes irreversible loss of precision.\n")


# --- 2. SHELL-DOMAIN SCALING (SDS — INTEGER NUMERATOR PRESERVATION) ---
Z_IN = Fraction("123456789.12345678")
MULTIPLIER = 7
Z_NUM = Z_IN.numerator
DEN = Z_IN.denominator
SDS_K_SHELL = 2 # Placeholder for the shell power

def sds_structural_test(z_in_num: int, multiplier: int, den: int, k_shell: int):
    # This test simulates how SDS handles the numerator (N) and denominator (D).
    # The calculation happens on the BigInt N, while D is stored in the shell structure.

    # 1. Encoding (The number enters the SDS system)
    # The initial numerator is encoded into the shell domain before calculation.
    _ = sns_number_integer(z_in_num, k_shell)

    # 2. Forward Operation (Scale): Multiply the BigInt numerator
    n_forward = z_in_num * multiplier

    # 3. Intermediate Encoding (The result is re-encoded before the reverse operation)
    encoded_result = sns_number_integer(n_forward, k_shell)

    # 4. Reverse Operation (Unscale): Divide the BigInt numerator component
    #    The division MUST be exact to prove reversibility: (N * M) / M = N
    n_reverse = n_forward // multiplier

    # 5. Decode: Convert the resulting BigInt back into a Fraction (Rational)
    z_reverse_fraction = Fraction(n_reverse, den)

    # The drift is calculated in the exact rational domain (should be 0).
    drift_fraction = z_reverse_fraction - Fraction(z_in_num, den)

    return z_reverse_fraction, drift_fraction, encoded_result

# Run the test, getting the recovered Fraction, drift, and the encoded result tuple
z_rev_sds, drift_sds, encoded_result = sds_structural_test(Z_NUM, MULTIPLIER, DEN, SDS_K_SHELL)

# Reassemble for comparison (using the recovered Fraction object)
delta_float = float(z_rev_sds) - float(Z_IN)

print("--- 2. SHELL-DOMAIN SCALING (SDS — INTEGER NUMERATOR PRESERVATION) ---")
print(f"Initial Z (Fraction): {Z_IN}")
print(f"Numerator (BigInt): {Z_NUM}")
print(f"Encoded Result (k, r, norm): {encoded_result}") # Show the intermediate SDS encoding
print(f"Denominator: {DEN}")
print(f"Recovered Z (Fraction): {z_rev_sds}")
print(f"Drift (Fraction): {drift_sds}")
print(f"Recovered Z (Float for Display): {float(z_rev_sds)}")
print(f"Drift from Original (Float): {delta_float}")


if drift_sds == 0:
    print("Interpretation: SDS successfully preserved the value and returned the exact Fraction (zero drift).")
else:
    print("Interpretation: SDS failed (drift in the BigInt numerator, unexpected).")


--- 1. IEEE 754 DOUBLE-PRECISION (CONTROL) ---
Initial Z: 123456789.12345678
Result (Float Reverse): 123456789.12345679
Drift: 1.4901161193847656e-08
Interpretation: Rounding drift causes irreversible loss of precision.

--- 2. SHELL-DOMAIN SCALING (SDS — INTEGER NUMERATOR PRESERVATION) ---
Initial Z (Fraction): 6172839456172839/50000000
Numerator (BigInt): 6172839456172839
Encoded Result (k, r, norm): (2, 4983, 43209876193209873)
Denominator: 50000000
Recovered Z (Fraction): 6172839456172839/50000000
Drift (Fraction): 0
Recovered Z (Float for Display): 123456789.12345678
Drift from Original (Float): 0.0
Interpretation: SDS successfully preserved the value and returned the exact Fraction (zero drift).


In [25]:
# @title Section 9: Extra Test — Representation-Dependent Mandelbrot Boundary Drift
# Section 9: Extra Test — Representation-Dependent Mandelbrot Boundary Drift
# ----------------------------------------------------------------------------------
# Purpose:
#   Demonstrate that near a sensitive Mandelbrot boundary, IEEE 754 floating-point
#   arithmetic and SDS integer-scaled arithmetic can follow different numerical
#   trajectories and escape at different iteration counts.
#
# Important:
#   This test does NOT claim that SDS prevents escape or reaches the maximum
#   iteration limit. The chosen point is slightly outside the boundary and is
#   expected to escape. The purpose is to show representation-dependent trajectory
#   divergence near a sensitive numerical boundary.
#
# Setup:
#   - Iteration limit: 50,000,000 steps.
#   - Test point: C = 0.2500000000001 + 0i.
#   - Float uses standard IEEE 754 complex iteration.
#   - SDS uses integer-scaled fixed-domain iteration.
#
# Expected Behavior:
#   - Both methods are expected to escape.
#   - The escape iteration may differ because the numerical representations follow
#     slightly different trajectories near the boundary.
#
# Results from this run:
#   Float Escape Iteration: 9,935,817
#   SDS Escape Iteration:   9,934,586
#
# Interpretation:
#   The float and SDS versions escape at different iterations. This demonstrates
#   that near a sensitive boundary, numerical representation can affect trajectory
#   behavior. The result supports the SDS argument that integer-scaled arithmetic
#   exposes and controls representation effects differently than IEEE 754
#   floating-point arithmetic.


# --- I. PARAMETERS ---
MAX_ITER_BONUS = 50000000
DEN = 10**128

# Escaping point highly sensitive to trajectory drift
C_FLOAT_BONUS = complex(0.2500000000001, 0)

# Exact BigInt Numerators for C = 0.2500000000001
C_RE_INT_BONUS = 2500000000001 * (10**(len(str(DEN)) - 14))
C_IM_INT_BONUS = 0


# --- II. ITERATION FUNCTIONS (Assuming they are defined and correct) ---
def mandelbrot_float(c, max_iter):
    # Standard IEEE 754 Double-Precision iteration.
    z = 0 + 0j
    for i in range(max_iter):
        z = z**2 + c
        if abs(z) > 2:
            return i
    return max_iter


# --- III. EXECUTE TEST ---
float_result = mandelbrot_float(C_FLOAT_BONUS, MAX_ITER_BONUS)
sds_result = mandelbrot_sds(C_RE_INT_BONUS, C_IM_INT_BONUS, DEN, MAX_ITER_BONUS)


# --- IV. OUTPUT ---
print("--- SECTION 9: Trajectory Drift Disparity ---")
print("--------------------------------------------------")
print(f"Test Point (C): {C_FLOAT_BONUS}")
print(f"Max Iterations: {MAX_ITER_BONUS:,}")

print("\n--- RESULTS ---")
print(f"Float Escape Iteration: {float_result:,}")
print(f"SDS Escape Iteration:   {sds_result:,}")
print("-" * 50)

print("\n--- FINAL INTERPRETATION ---")
if float_result != sds_result:
    print("Result: float and SDS escape at different iterations.")
    print(f"Float escaped at iteration {float_result:,}.")
    print(f"SDS escaped at iteration {sds_result:,}.")
    print("Interpretation: this demonstrates representation-dependent trajectory divergence near a sensitive boundary.")
else:
    print("Result: both methods escaped at the same iteration.")
    print("Interpretation: this point does not demonstrate representation-dependent trajectory divergence.")




--- SECTION 9: Trajectory Drift Disparity ---
--------------------------------------------------
Test Point (C): (0.2500000000001+0j)
Max Iterations: 50,000,000

--- RESULTS ---
Float Escape Iteration: 9,935,817
SDS Escape Iteration:   9,934,586
--------------------------------------------------

--- FINAL INTERPRETATION ---
Result: float and SDS escape at different iterations.
Float escaped at iteration 9,935,817.
SDS escaped at iteration 9,934,586.
Interpretation: this demonstrates representation-dependent trajectory divergence near a sensitive boundary.
